# Create IATI Awards (generic: any IATI publisher)

One notebook for every aid agency that publishes its activities in the IATI
standard (Sida, Norad / Norway MFA, FCDO, and later BMZ, Danida, Netherlands
MFA, AFD ...). It generalises `CreateIDRCAwards`: the same source format, but
publisher-agnostic, research-filtered, and it keeps what the IDRC notebook
threw away (every recipient country with its percentage, regions, every
participating organisation with its role, sector vocabularies, transactions
with provider/receiver organisations, parent/child links) in `iati_raw`.

**Prerequisites**
- Run `scripts/local/iati_to_s3.py --publisher <ref> [...]` (or `--all`). It
  reads the publisher's files from IATI's Bulk Data Service
  (https://bulk-data.iatistandard.org, no key), parses them, applies the
  research filter and uploads ONE parquet per publisher:
  `s3://openalex-ingest/awards/iati_<publisher>/iati_<publisher>_projects.parquet`
  (runbook §1.4 shrink guard).
- The publisher must have a row in `scripts/local/iati_publishers.csv`
  (`publisher_ref, openalex_funder_id, funder_name, research_filter, ...`).
  **Adding a publisher is a CSV row plus one script run. This notebook does not
  change.**

**Research filter (required).** IATI is all aid, not only research. Only
activities admitted by a rule in `scripts/local/iati_research_rules.csv` reach
S3, and so this notebook. The rules are data: the OECD DAC purpose codes whose
name says research (11182, 12182, 12382, 23182/23082, 31182, 31282, 31382,
32182, 41082, 43082) when they make up at least `min_research_share` percent
(default 50) of the activity's sector allocation, plus per-publisher programme
rules (Sida's "Research cooperation" strategy; Norad's NORHED). Each row
carries `research_rule_ids`, the rule(s) that admitted it. Cell "Step 1.5"
fails the run if any row arrives without one.

**Grain (`award_level` in the CSV).** `leaf`: each activity that carries money
is an award and its parent programme's title becomes `funder_scheme` (FCDO
components). `top`: each top-level activity is an award and its children roll
up into it (a Sida contribution is cut into hundreds of country/year slices).

**`funder_award_id` (§2.1.1).** The IATI identifier with the publisher prefix
removed (`award_id_regex` in the CSV), because that is what papers cite:
`54100113` (Sida contribution), `QZA-21/0082` (Norad agreement),
`204074-101` (FCDO component). The full `iati_identifier` stays in `iati_raw`.
Checked against the award ids already cited for these funders in
`openalex_awards_raw`; see the stub-overlap cell at the end.

**Funder mapping (§2.3.2).** A publisher file can carry several funders
(Norway's holds Norad, the MFA and other ministries; Sweden's holds Sida, the
MFA, the Swedish Research Council ...). The script routes each activity by its
extending/accountable organisation (`funder_org_refs` in the CSV); activities
that match no row are not uploaded. Canonical funder metadata is read from
`openalex.funders.funders` by id here.

**Amounts.** `amount` = the first positive of the publisher's `amount_basis`
order (outgoing commitments, budget, disbursements + expenditure), in the
currency the publisher reports (SEK, NOK, GBP ...). Budgets: a revised budget
replaces the original for the same period. No currency conversion here.

**Recipient.** IATI funds organisations, not people: `lead_investigator`
carries only the affiliation (the organisation that receives the largest
commitments/disbursements, else the first implementing organisation).
`affiliation.country` is the country the organisation is REGISTERED in, read
from its IATI organisation identifier (`GB-COH-...` -> GB), else NULL. It is
never the activity's recipient country. Recipient countries stay in
`iati_raw.recipient_countries_json` / `recipient_country_codes`.

**Activity level or transaction level.** IATI lets a publisher put recipient
countries/regions and sectors on the activity OR on its transactions (US
agencies, Elrha, GARDP, Grand Challenges Canada, LSE and Sida's level-2 slices
use transactions). The script reads whichever level was used and writes
`derived_recipient_countries_json`, `derived_recipient_regions_json` and
`derived_sectors_json` (percentage shares; transaction-level shares are
weighted by transaction value) with `recipient_geo_level` / `sector_level` =
`activity`, `transaction`, `children` (rolled up) or `parent` (inherited). The
research filter uses the same transaction-level sector shares.

**Provenance:** `iati_<publisher>` (e.g. `iati_sida`), one per publisher.

**Priority / publishing.** The priority integer is assigned by the step-0
coordinator (oxjob #1451). Until it is, and until someone sets the two job
parameters, **this notebook does NOT write to `openalex_awards_raw`**: it
builds `openalex.awards.iati_raw` and `openalex.awards.iati_awards` only.
To publish, run with `publish_to_awards_raw=true` and `priority=<N>`.

## Step 1: Create staging table from S3 (all IATI publishers)

In [ ]:
# Every publisher parquet comes from the same script and has the same all-string
# columns, so one glob reads them all. mergeSchema keeps the read working if a
# publisher's file was written by an older version of the script.
from pyspark.sql import functions as F

RAW_GLOB = "s3a://openalex-ingest/awards/iati_*/iati_*_projects.parquet"

raw = (
    spark.read.option("mergeSchema", "true").parquet(RAW_GLOB)
    .withColumn("databricks_ingested_at", F.current_timestamp())
)
(
    raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("openalex.awards.iati_raw")
)
print(f"openalex.awards.iati_raw: {spark.table('openalex.awards.iati_raw').count():,} rows")

In [ ]:
%sql
SELECT provenance, publisher_short_name, openalex_funder_id, award_level, COUNT(*) AS activities
FROM openalex.awards.iati_raw
GROUP BY 1, 2, 3, 4
ORDER BY 1, 5 DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.iati_raw;

In [ ]:
%sql
SELECT iati_identifier, funder_award_id, title, hierarchy, award_amount, award_currency, award_amount_basis,
       award_start_date, award_end_date, lead_org_name, recipient_country_codes, research_rule_ids
FROM openalex.awards.iati_raw LIMIT 10;

## Step 1.5: Research-filter guard and amount sanity check
Nothing may reach the awards table unless the research filter admitted it. The
upload script only writes admitted rows; this cell fails the run if that ever
stops being true (for example a hand-uploaded parquet).

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 0,
    'iati_raw holds rows that the research filter did not admit; re-run scripts/local/iati_to_s3.py'
) AS research_filter_guard
FROM openalex.awards.iati_raw
WHERE COALESCE(is_research, 'false') != 'true'
   OR COALESCE(ship, 'false') != 'true'
   OR research_rule_ids IS NULL
   OR research_rule_ids = '[]';

In [ ]:
%sql
-- Which rule admitted how many activities, per publisher
SELECT provenance, rule_id, COUNT(*) AS activities
FROM openalex.awards.iati_raw
LATERAL VIEW explode(from_json(research_rule_ids, 'array<string>')) t AS rule_id
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

In [ ]:
%sql
-- Amount-shape and currency per publisher BEFORE the transformation (§1.5)
SELECT
    provenance,
    COUNT(*) AS rows,
    COUNT(award_amount) AS has_amount,
    collect_set(award_currency) AS currencies,
    collect_set(award_amount_basis) AS amount_basis,
    ROUND(MIN(TRY_CAST(award_amount AS DOUBLE)), 0) AS min_amount,
    ROUND(MAX(TRY_CAST(award_amount AS DOUBLE)), 0) AS max_amount,
    ROUND(AVG(TRY_CAST(award_amount AS DOUBLE)), 0) AS avg_amount
FROM openalex.awards.iati_raw
GROUP BY 1 ORDER BY 1;

## Step 1.6: Funder existence check
Every `openalex_funder_id` in the staging table must resolve to exactly one row
in `openalex.funders.funders`. If not, STOP and fix `iati_publishers.csv`.

In [ ]:
%sql
SELECT r.openalex_funder_id, f.display_name, f.ror_id, f.doi, f.country_code, r.activities
FROM (
    SELECT openalex_funder_id, COUNT(*) AS activities
    FROM openalex.awards.iati_raw GROUP BY 1
) r
LEFT JOIN openalex.funders.funders f
  ON f.funder_id = TRY_CAST(r.openalex_funder_id AS BIGINT)
ORDER BY r.activities DESC;

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 0,
    'an openalex_funder_id in iati_raw does not resolve to exactly one row of openalex.funders.funders'
) AS funder_guard
FROM (
    SELECT r.openalex_funder_id
    FROM (SELECT DISTINCT openalex_funder_id FROM openalex.awards.iati_raw) r
    LEFT JOIN openalex.funders.funders f
      ON f.funder_id = TRY_CAST(r.openalex_funder_id AS BIGINT)
    GROUP BY r.openalex_funder_id
    HAVING COUNT(f.funder_id) != 1
);

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.iati_awards
USING delta
AS
WITH src AS (
    SELECT
        r.*,
        TRY_CAST(r.openalex_funder_id AS BIGINT) AS funder_id_num,
        TRIM(r.funder_award_id) AS award_id,
        TRY_CAST(r.award_amount AS DOUBLE) AS amount_num,
        TRY_TO_DATE(r.award_start_date, 'yyyy-MM-dd') AS d_start,
        TRY_TO_DATE(r.award_end_date, 'yyyy-MM-dd') AS d_end
    FROM openalex.awards.iati_raw r
    -- Defence in depth: the research filter is re-applied here.
    WHERE r.ship = 'true'
      AND r.is_research = 'true'
      AND r.research_rule_ids IS NOT NULL
      AND r.funder_award_id IS NOT NULL
      AND TRIM(r.funder_award_id) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(s.award_id)))) % 9000000000 AS id,
    NULLIF(TRIM(s.title), '') AS display_name,
    NULLIF(TRIM(s.description), '') AS description,
    f.funder_id,
    s.award_id AS funder_award_id,
    -- Amount only when positive (zero / negative totals are placeholders or reversals)
    CASE WHEN s.amount_num > 0 THEN s.amount_num END AS amount,
    CASE WHEN s.amount_num > 0 THEN s.award_currency END AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    -- leaf grain: the parent programme's title is the scheme (FCDO programme -> component)
    NULLIF(TRIM(s.parent_title), '') AS funder_scheme,
    s.provenance AS provenance,
    -- 1900-01-01 style placeholders are not dates
    CASE WHEN YEAR(s.d_start) >= 1950 THEN s.d_start END AS start_date,
    CASE WHEN YEAR(s.d_end) >= 1950 THEN s.d_end END AS end_date,
    CASE WHEN YEAR(s.d_start) >= 1950 THEN YEAR(s.d_start) END AS start_year,
    CASE WHEN YEAR(s.d_end) >= 1950 THEN YEAR(s.d_end) END AS end_year,
    -- IATI funds organisations: affiliation only. country = where the organisation is
    -- registered (from its IATI org identifier), never the activity's recipient country.
    CASE WHEN NULLIF(TRIM(s.lead_org_name), '') IS NOT NULL THEN struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            TRIM(s.lead_org_name) AS name,
            s.lead_org_country AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    s.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(s.award_id)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM src s
JOIN openalex.funders.funders f ON f.funder_id = s.funder_id_num;

## Step 3: Insert into openalex_awards_raw — OFF by default
`openalex_awards_raw` goes public with the next awards rebuild, so this cell
only writes when BOTH job parameters are set: `publish_to_awards_raw=true` and
`priority=<the integer assigned in CreateAwards.ipynb>`. With the defaults it
prints SKIPPED and the notebook ends having built only `iati_raw` and
`iati_awards`. All `iati_<publisher>` provenances share the one priority slot.

In [ ]:
dbutils.widgets.text("publish_to_awards_raw", "false")
dbutils.widgets.text("priority", "")

publish = dbutils.widgets.get("publish_to_awards_raw").strip().lower() == "true"
priority_raw = dbutils.widgets.get("priority").strip()

if not publish:
    print("SKIPPED: publish_to_awards_raw is not 'true'. "
          "openalex.awards.iati_awards was built; nothing was written to openalex_awards_raw.")
else:
    if not priority_raw.isdigit() or int(priority_raw) < 3:
        raise ValueError(
            f"priority={priority_raw!r}: pass the integer reserved for IATI in CreateAwards.ipynb "
            "(a direct ingest must be >= 3; 0-2 are citation shells)")
    priority = int(priority_raw)

    # Remove previous data for this source before inserting fresh data.
    spark.sql(f'''
        DELETE FROM openalex.awards.openalex_awards_raw
        WHERE provenance RLIKE '^iati_[a-z0-9_]+$' AND priority = {priority}
    ''')
    spark.sql(f'''
        INSERT INTO openalex.awards.openalex_awards_raw
        SELECT
            id, display_name, description, funder_id, funder_award_id, amount, currency,
            funder, funding_type, funder_scheme, provenance, start_date, end_date,
            start_year, end_year, lead_investigator, co_lead_investigator, investigators,
            landing_page_url, doi, works_api_url, created_date, updated_date,
            {priority} as priority
        FROM openalex.awards.iati_awards
    ''')
    print(f"Inserted openalex.awards.iati_awards into openalex_awards_raw at priority {priority}.")

## Verification

In [ ]:
%sql
-- 6.1 counts: awards must equal staged rows; ids must be unique
SELECT
    (SELECT COUNT(*) FROM openalex.awards.iati_raw) AS staged_rows,
    COUNT(*) AS awards,
    COUNT(DISTINCT id) AS distinct_ids,
    COUNT(DISTINCT funder_id, LOWER(funder_award_id)) AS distinct_funder_award_ids
FROM openalex.awards.iati_awards;

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids in iati_awards') AS unique_id_guard
FROM openalex.awards.iati_awards;

In [ ]:
%sql
-- 6.3 completeness and 6.7 amount/currency, per publisher
SELECT
    provenance,
    COUNT(*) AS total,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    ROUND(try_divide(COUNT(end_date), COUNT(*)) * 100.0, 1) AS pct_end_date,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount,
    ROUND(SUM(amount), 0) AS sum_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_recipient_org,
    ROUND(try_divide(COUNT(funder_scheme), COUNT(*)) * 100.0, 1) AS pct_scheme
FROM openalex.awards.iati_awards
GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
-- 6.4 sample for visual review
SELECT provenance, funder_award_id, SUBSTRING(display_name, 1, 80) AS title, funder_scheme,
       amount, currency, start_date, end_date,
       lead_investigator.affiliation.name AS recipient_org,
       lead_investigator.affiliation.country AS recipient_org_country,
       landing_page_url
FROM openalex.awards.iati_awards
ORDER BY xxhash64(CAST(id AS STRING)) LIMIT 30;

In [ ]:
%sql
-- 6.4a frequency check. There are no person names here (given/family are NULL
-- by design), so the check is on recipient organisations and titles: the top
-- organisations should be real research performers / agencies, never a channel
-- category such as 'International NGO' or a placeholder such as 'Other'.
SELECT provenance, lead_investigator.affiliation.name AS recipient_org, COUNT(*) AS n
FROM openalex.awards.iati_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 40;

In [ ]:
%sql
SELECT provenance, display_name, COUNT(*) AS n
FROM openalex.awards.iati_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 0, 'lead_investigator carries a person name; IATI awards must be affiliation-only'
) AS no_person_names_guard
FROM openalex.awards.iati_awards
WHERE lead_investigator.given_name IS NOT NULL OR lead_investigator.family_name IS NOT NULL;

In [ ]:
%sql
-- 6.5 funder consistency: one publisher file can carry several funders (§2.3.2)
SELECT provenance, funder.display_name, funder_id, COUNT(*) AS awards
FROM openalex.awards.iati_awards
GROUP BY 1, 2, 3 ORDER BY 1, 4 DESC;

In [ ]:
%sql
-- 6.6 year distribution per publisher
SELECT start_year, provenance, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS sum_amount
FROM openalex.awards.iati_awards
WHERE start_year IS NOT NULL
GROUP BY 1, 2 ORDER BY 1 DESC, 2;

In [ ]:
%sql
-- §2.1.1 shape check. `full_iati_form` counts awards whose shipped id is still the
-- whole IATI identifier: expected 0 for a publisher with an `award_id_regex`
-- (Sida, Norad), and equal to the identifiers the regex does not cover otherwise
-- (FCDO: only the ex-FCO `GB-GOV-3-...` activities).
SELECT
    a.provenance,
    COUNT(*) AS awards,
    COUNT_IF(a.funder_award_id = r.iati_identifier) AS full_iati_form,
    MIN(a.funder_award_id) AS example_min,
    MAX(a.funder_award_id) AS example_max
FROM openalex.awards.iati_awards a
JOIN openalex.awards.iati_raw r
  ON r.provenance = a.provenance AND TRIM(r.funder_award_id) = a.funder_award_id
GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
-- Overlap with the award ids papers already cite for these funders (citation
-- shells at priority 0/1): how many existing stub ids this ingest collapses onto.
SELECT
    a.provenance,
    a.funder_id,
    COUNT(DISTINCT s.id) AS stub_ids_upgraded
FROM openalex.awards.iati_awards a
JOIN openalex.awards.openalex_awards_raw s
  ON s.id = a.id AND s.priority < 3
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
-- Recipient countries are NOT in the awards schema; they stay in iati_raw.
-- Every recipient country with its share, one row per (award, country), and the
-- level the publisher stated it at (activity / transaction / children / parent):
SELECT r.provenance, r.funder_award_id, r.recipient_geo_level, r.recipient_geo_share_basis,
       c.code AS recipient_country, c.percentage
FROM openalex.awards.iati_raw r
LATERAL VIEW explode(from_json(r.derived_recipient_countries_json,
    'array<struct<code:string,percentage:double>>')) t AS c
LIMIT 20;

In [ ]:
%sql
-- Which level each publisher states geography and sectors at (shipped awards)
SELECT provenance, recipient_geo_level, sector_level, COUNT(*) AS awards
FROM openalex.awards.iati_raw
GROUP BY 1, 2, 3 ORDER BY 1, 4 DESC;

In [ ]:
%sql
-- Every participating organisation with its role (1 funding, 2 accountable,
-- 3 extending, 4 implementing), one row per (award, organisation):
SELECT r.provenance, r.funder_award_id, o.role, o.type, o.ref, o.name
FROM openalex.awards.iati_raw r
LATERAL VIEW explode(from_json(r.participating_orgs_json,
    'array<struct<role:string,type:string,ref:string,name:string,activity_id:string,crs_channel_code:string>>')) t AS o
LIMIT 20;

In [ ]:
%sql
-- 6.8 rows in the shared raw table. Expect ZERO rows until the notebook is run
-- with publish_to_awards_raw=true and a priority.
SELECT provenance, priority, COUNT(*) AS n
FROM openalex.awards.openalex_awards_raw
WHERE provenance RLIKE '^iati_[a-z0-9_]+$'
GROUP BY 1, 2 ORDER BY 1;